In [1]:
import matplotlib.pyplot as plt
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from dotenv import load_dotenv

/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

In [3]:
load_dotenv()

True

In [4]:
dataset = load_dataset('imdb')
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
model_name = "distilbert/distilbert-base-uncased"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
sample_tokens = tokenizer('This is it.')
sample_tokens

In [ ]:
tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'])

In [ ]:
def get_token_lengths(texts, tokenizer):
    lengths = []
    for text in texts:
        tokens = tokenizer(
            text,
            truncate = False,
            padding = False,
            add_special_tokens=True 
        )
        lengths.append(len(tokens['input_ids']))
    return lengths

In [ ]:
shuffled_dataset = dataset['train'].shuffle(seed=42)

In [ ]:
num = 10000
texts = list(shuffled_dataset['text'][:num])
token_lengths = get_token_lengths(texts, tokenizer)

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(token_lengths, bins = 50)
plt.xlabel('Number of tokens per review')
plt.ylabel('Number of reviews')
plt.title('Review length distribution (tokens)')
plt.show()

In [ ]:
for p in [50, 75, 90, 95, 99]:
    print(f"{p}th percentile: {np.percentile(token_lengths, p):.0f} tokens")

In [7]:
def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation = True,
        padding = 'max_length',
        max_length=256
    )

In [ ]:
example_review = dataset['train'][0]
example_review

In [ ]:
example_review_num_words = len(example_review['text'].split(' '))
example_review_num_words

In [ ]:
example_token = tokenize_function(example_review)
example_token

In [ ]:
len(example_token['input_ids'])

In [ ]:
decoded_review = tokenizer.decode(
    example_token['input_ids'],
    skip_special_tokens=True 
)
print(decoded_review)

As expected, some trailing part of the review is truncated. 

In [ ]:
tokenized_batch = tokenize_function(dataset['train'][:5])

In [ ]:
len(tokenized_batch['input_ids'])

In [8]:
encoded_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

Map: 100%|██████████| 50000/50000 [00:05<00:00, 9897.19 examples/s] 


In [9]:
encoded_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2394.75it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model)

In [12]:
sample_train_set = encoded_dataset['train'].train_test_split(
    test_size = 0.01,
    seed = 42
)['test']

In [13]:
sample_test_set = encoded_dataset['test'].train_test_split(
    test_size = 0.01,
    seed = 42
)['test']

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [16]:
training_args = TrainingArguments(
    output_dir='../models/test-run-1',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
)

In [17]:
trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=sample_train_set,
    eval_dataset=sample_test_set,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.664594,0.704000
2,No log,0.589128,0.788000
3,No log,0.505898,0.820000
4,No log,0.477431,0.820000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]
/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.79it/s]
/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]
/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now

TrainOutput(global_step=64, training_loss=0.5479109287261963, metrics={'train_runtime': 56.3742, 'train_samples_per_second': 17.739, 'train_steps_per_second': 1.135, 'total_flos': 66233699328000.0, 'train_loss': 0.5479109287261963, 'epoch': 4.0})

In [19]:
trainer.evaluate()

/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.4774308502674103,
 'eval_accuracy': 0.82,
 'eval_runtime': 2.7717,
 'eval_samples_per_second': 90.198,
 'eval_steps_per_second': 5.773,
 'epoch': 4.0}

In [20]:
trainer.state.log_history

[{'eval_loss': 0.6645936369895935,
  'eval_accuracy': 0.704,
  'eval_runtime': 2.7773,
  'eval_samples_per_second': 90.015,
  'eval_steps_per_second': 5.761,
  'epoch': 1.0,
  'step': 16},
 {'eval_loss': 0.5891279578208923,
  'eval_accuracy': 0.788,
  'eval_runtime': 2.7738,
  'eval_samples_per_second': 90.129,
  'eval_steps_per_second': 5.768,
  'epoch': 2.0,
  'step': 32},
 {'eval_loss': 0.5058978796005249,
  'eval_accuracy': 0.82,
  'eval_runtime': 2.7793,
  'eval_samples_per_second': 89.949,
  'eval_steps_per_second': 5.757,
  'epoch': 3.0,
  'step': 48},
 {'eval_loss': 0.4774308502674103,
  'eval_accuracy': 0.82,
  'eval_runtime': 2.7725,
  'eval_samples_per_second': 90.17,
  'eval_steps_per_second': 5.771,
  'epoch': 4.0,
  'step': 64},
 {'train_runtime': 56.3742,
  'train_samples_per_second': 17.739,
  'train_steps_per_second': 1.135,
  'total_flos': 66233699328000.0,
  'train_loss': 0.5479109287261963,
  'epoch': 4.0,
  'step': 64},
 {'eval_loss': 0.4774308502674103,
  'eval_ac